# NLP Tasks: PDF, DOCX and API Data Processing

This notebook contains solutions to all 9 tasks, with each question written above its code cell.

**How to run:** run the *Setup* cells first. They create sample files (`sample.pdf`, `doc_a.docx`, `doc_b.docx`) so every task runs without any uploads. To use your own files, change `PDF_PATH`, `DOCX_A` and `DOCX_B` in the setup cell. Tasks 7-9 need internet access. Tasks 1 and 2 ask for input via `input()`.

## Setup

# NLP Lab Task 02
**Name / Reg No:** Urva Ishfaq/FA23-BAI-051


In [2]:
!pip install pdfplumber python-docx pandas nltk requests reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 83.4 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [3]:
# Run once if needed:
# !pip install pdfplumber python-docx pandas nltk requests reportlab
import re, csv, math, sys, json, pprint
from collections import Counter
import requests
import pandas as pd
import pdfplumber
import docx
from docx.text.paragraph import Paragraph
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

In [4]:
import os
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

PDF_PATH = "sample.pdf"
DOCX_A = "doc_a.docx"
DOCX_B = "doc_b.docx"

# ---------- sample PDF (6 pages, page 4 is intentionally empty) ----------
PAGE_TEXT = [
 "Natural language processing is a field of artificial intelligence. It helps computers understand human language. "
 "Visit https://example.com/nlp for more details. In 2023 over 500 papers were published. Language models learn patterns from text.",
 "Text preprocessing is an important step in every language pipeline. Tokenization splits text into words. "
 "Stopwords such as the and is are often removed. Cleaning text improves the quality of language models.",
 "Machine learning models need data. Data must be cleaned before training. A good pipeline handles punctuation, digits, and URLs like http://data.org/set1. "
 "The pipeline should be reproducible and well documented for every project.",
 "",
 "Word frequency analysis reveals the most common terms in a document. Frequency counts support search and classification. "
 "Researchers compare frequency tables across many different documents to find interesting patterns.",
 "In conclusion, language processing combines linguistics and computer science. Good preprocessing leads to better models. "
 "Future work will explore larger corpora and deeper neural architectures for text understanding.",
]
if not os.path.exists(PDF_PATH):
    c = canvas.Canvas(PDF_PATH, pagesize=A4)
    for txt in PAGE_TEXT:
        y = 800
        words, line = txt.split(), ""
        for w in words:
            if len(line) + len(w) > 85:
                c.drawString(40, y, line); y -= 16; line = ""
            line += w + " "
        if line: c.drawString(40, y, line)
        c.showPage()
    c.save()

# ---------- sample DOCX files ----------
def make_doc(path, headings, bodies, table_rows=None, intro=None):
    d = docx.Document()
    if intro:
        d.add_paragraph(intro)
    for h, b in zip(headings, bodies):
        d.add_heading(h, level=1)
        for para in b:
            d.add_paragraph(para)
        d.add_paragraph("")                       # blank paragraph
    if table_rows:
        t = d.add_table(rows=len(table_rows), cols=len(table_rows[0]))
        for i, r in enumerate(table_rows):
            for j, v in enumerate(r):
                t.cell(i, j).text = v
        d.add_paragraph("Paragraph after the table.")
    d.add_paragraph("A short note in body text style.", style="Body Text")
    d.add_paragraph("Quoted line for the quote style.", style="Quote")
    d.save(path)

shared = "This paragraph appears in both documents and should be removed once during deduplication."
if not os.path.exists(DOCX_A):
    make_doc(DOCX_A,
        ["Introduction", "Methods", "Results"],
        [["Natural language processing turns raw text into structured information that programs can use.", shared],
         ["We collected documents and cleaned them with regular expressions before analysis began.",
          "Each document was tokenized into words and counted using simple Python tools."],
         ["The word frequency tables show clear patterns across all of the documents we studied."]],
        table_rows=[["Name", "Role", "Score"], ["Ali", "Analyst", "90"], ["Sara", "", "85"], ["Omar", "Engineer", ""]],
        intro="This document describes a small study of text processing methods and tools.")
if not os.path.exists(DOCX_B):
    make_doc(DOCX_B,
        ["Introduction", "Discussion", "Conclusion"],
        [["Text analysis helps researchers discover trends hidden inside large collections of documents.", shared.upper()],
         ["The results suggest that careful preprocessing greatly improves downstream model quality overall."],
         ["Future studies should use bigger datasets and more advanced language models for better insight."]],
        intro="This second document compares several approaches to automatic text analysis.")
print("Sample files ready:", PDF_PATH, DOCX_A, DOCX_B)

Sample files ready: sample.pdf doc_a.docx doc_b.docx


## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

**(a)** Open a PDF file using PyPDF2 or pdfplumber and extract text from each page individually. Store the result as a list of strings where each index represents one page. Print the total number of pages extracted.

**(b)** Merge all page strings into a single string. Use `re.sub()` to remove all punctuation and digits, convert to lowercase, and split on whitespace to produce a token list. Print the total token count.

**(c)** Build a word frequency dictionary from the token list. Remove at least 20 common stopwords. Print the top 10 most frequent words along with their counts in descending order.

**(d)** Accept a keyword from the user via `input()`. Search through the page-wise list, split each page's text into sentences, and print every sentence containing the keyword along with its page number. If no match is found, print an appropriate message.

In [5]:
# (a)
with pdfplumber.open(PDF_PATH) as pdf:
    pages = [(page.extract_text() or "") for page in pdf.pages]
print("Total pages extracted:", len(pages))

Total pages extracted: 6


In [6]:
# (b)
full_text = " ".join(pages)
cleaned = re.sub(r"[^\w\s]|\d|_", "", full_text)   # remove punctuation and digits
tokens = cleaned.lower().split()
print("Total tokens:", len(tokens))

Total tokens: 152


In [7]:
# (c)
stop_words_t1 = {"the","is","a","an","and","of","to","in","it","for","on","with","as","are","be",
                 "by","at","that","this","from","or","such","often","was","were","has","have","will","should","must"}
freq = {}
for tok in tokens:
    if tok not in stop_words_t1:
        freq[tok] = freq.get(tok, 0) + 1
top10 = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("Top 10 words:")
for w, c in top10:
    print(f"  {w:<15}{c}")

Top 10 words:
  language       6
  text           5
  models         4
  pipeline       3
  frequency      3
  processing     2
  patterns       2
  preprocessing  2
  every          2
  data           2


In [8]:
# (d)
keyword = input("Enter a keyword to search: ").strip().lower()
found = False
for page_no, text in enumerate(pages, start=1):
    sentences = re.split(r"(?<=[.!?])\s+", text.replace("\n", " "))
    for s in sentences:
        if keyword and keyword in s.lower():
            print(f"[Page {page_no}] {s.strip()}")
            found = True
if not found:
    print(f"No sentence containing '{keyword}' was found.")

Enter a keyword to search: hello
No sentence containing 'hello' was found.


## Task 2: Structured Page Metadata Extraction and CSV Export

**(a)** Open a multi-page PDF (minimum 5 pages). For each page, build a dictionary with exactly these keys: `page_number`, `raw_text`, `word_count`, and `char_count` (excluding spaces). Collect all dictionaries into a list called `pages_data` and print the first 2 entries.

**(b)** From `pages_data`, identify and print: (i) the page with the highest word count and its full content, (ii) the page with the lowest non-zero word count, and (iii) the average word count across all pages rounded to 2 decimal places. Handle pages that return zero words.

**(c)** Accept a search phrase via `input()`. Perform a case-insensitive search across all pages. For every page containing the phrase, print its page number and how many times the phrase appears. If no page contains the phrase, raise a custom exception named `PhraseNotFoundError` and catch it gracefully.

**(d)** Using Python's built-in `csv` module (not pandas), write a file named `pdf_summary.csv` with columns: `page_number`, `word_count`, `char_count`. Then re-read that file and print its contents row by row to confirm the write was successful.

In [9]:
# (a)
pages_data = []
with pdfplumber.open(PDF_PATH) as pdf:
    assert len(pdf.pages) >= 5, "PDF must have at least 5 pages"
    for i, page in enumerate(pdf.pages, start=1):
        text = page.extract_text() or ""
        pages_data.append({
            "page_number": i,
            "raw_text": text,
            "word_count": len(text.split()),
            "char_count": len(re.sub(r"\s", "", text)),   # excludes spaces/whitespace
        })
for entry in pages_data[:2]:
    print(entry, "\n")

{'page_number': 1, 'raw_text': 'Natural language processing is a field of artificial intelligence. It helps computers\nunderstand human language. Visit https://example.com/nlp for more details. In 2023\nover 500 papers were published. Language models learn patterns from text.', 'word_count': 33, 'char_count': 210} 

{'page_number': 2, 'raw_text': 'Text preprocessing is an important step in every language pipeline. Tokenization\nsplits text into words. Stopwords such as the and is are often removed. Cleaning text\nimproves the quality of language models.', 'word_count': 32, 'char_count': 176} 



In [10]:
# (b)
max_page = max(pages_data, key=lambda p: p["word_count"])
print(f"(i) Highest word count: page {max_page['page_number']} ({max_page['word_count']} words)")
print(max_page["raw_text"])

non_zero = [p for p in pages_data if p["word_count"] > 0]
if non_zero:
    min_page = min(non_zero, key=lambda p: p["word_count"])
    print(f"\n(ii) Lowest non-zero word count: page {min_page['page_number']} ({min_page['word_count']} words)")
else:
    print("\n(ii) All pages are empty.")

empty_pages = [p["page_number"] for p in pages_data if p["word_count"] == 0]
if empty_pages:
    print("Pages with zero words (handled):", empty_pages)

avg = round(sum(p["word_count"] for p in pages_data) / len(pages_data), 2) if pages_data else 0
print(f"(iii) Average word count: {avg}")

(i) Highest word count: page 1 (33 words)
Natural language processing is a field of artificial intelligence. It helps computers
understand human language. Visit https://example.com/nlp for more details. In 2023
over 500 papers were published. Language models learn patterns from text.

(ii) Lowest non-zero word count: page 6 (28 words)
Pages with zero words (handled): [4]
(iii) Average word count: 25.67


In [ ]:
# (c)
class PhraseNotFoundError(Exception):
    pass

phrase = input("Enter a search phrase: ").strip().lower()
try:
    hits = []
    for p in pages_data:
        n = p["raw_text"].lower().count(phrase) if phrase else 0
        if n:
            hits.append((p["page_number"], n))
    if not hits:
        raise PhraseNotFoundError(f"The phrase '{phrase}' was not found in any page.")
    for page_no, n in hits:
        print(f"Page {page_no}: '{phrase}' appears {n} time(s)")
except PhraseNotFoundError as e:
    print("PhraseNotFoundError caught:", e)

In [ ]:
# (d)
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for p in pages_data:
        writer.writerow([p["page_number"], p["word_count"], p["char_count"]])

print("Contents of pdf_summary.csv:")
with open("pdf_summary.csv", newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(row)

## Task 3: PDF Text Preprocessing Pipeline for NLP

**(a)** Open any PDF file and extract all text into one string. Apply the following cleaning steps in order using `re.sub()`: (i) remove all URLs, (ii) remove all digits, (iii) remove all non-alphabetic characters except spaces and full stops, (iv) collapse multiple consecutive spaces into a single space. Print the character count before and after cleaning.

**(b)** Convert the cleaned text to lowercase. Split into a list of sentences using `re.split(r'[.!?]', text)`. Strip each sentence of leading and trailing whitespace. Remove empty strings. Print the sentence count before and after removing empty strings.

**(c)** Tokenize each sentence into words. Remove English stopwords using `nltk.corpus.stopwords.words('english')`. Discard any sentence that has fewer than 5 tokens remaining after stopword removal. Print the sentence count before and after this filter and display 3 sample sentences.

**(d)** Write all remaining sentences to a file named `nlp_ready_output.txt`, one sentence per line. Re-read the file and print: (i) total lines written, (ii) the longest sentence by character length, and (iii) the shortest sentence by character length.

In [ ]:
# (a)
with pdfplumber.open(PDF_PATH) as pdf:
    raw = " ".join((p.extract_text() or "") for p in pdf.pages)

before_len = len(raw)
t = re.sub(r"https?://\S+|www\.\S+", "", raw)      # (i) URLs
t = re.sub(r"\d", "", t)                            # (ii) digits
t = re.sub(r"[^a-zA-Z\s.]", "", t)                  # (iii) keep letters, spaces, full stops
t = re.sub(r"\s+", " ", t)                          # (iv) collapse spaces
print("Characters before cleaning:", before_len)
print("Characters after cleaning: ", len(t))

In [ ]:
# (b)
t = t.lower()
sentences_all = re.split(r"[.!?]", t)
print("Sentence count before removing empties:", len(sentences_all))
sentences = [s.strip() for s in sentences_all]
sentences = [s for s in sentences if s]
print("Sentence count after removing empties: ", len(sentences))

In [ ]:
# (c)
stop_nltk = set(stopwords.words("english"))
print("Sentences before filter:", len(sentences))
filtered = []
for s in sentences:
    toks = [w for w in s.split() if w not in stop_nltk]
    if len(toks) >= 5:
        filtered.append(s)
print("Sentences after filter: ", len(filtered))
print("\n3 sample sentences:")
for s in filtered[:3]:
    print(" -", s)

In [ ]:
# (d)
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for s in filtered:
        f.write(s + "\n")

with open("nlp_ready_output.txt", encoding="utf-8") as f:
    lines = [ln.rstrip("\n") for ln in f]
print("(i) Total lines written:", len(lines))
if lines:
    print("(ii) Longest sentence: ", max(lines, key=len))
    print("(iii) Shortest sentence:", min(lines, key=len))

## Task 4: DOCX Paragraph Classification and Statistical Analysis

**(a)** Open any `.docx` file using python-docx. For each paragraph, store its text and `style.name` in a list of dictionaries. Print the total paragraph count and all unique style names found in the document.

**(b)** Classify all paragraphs into exactly three groups: (i) **Headings**: style name starts with "Heading", (ii) **Body**: style is "Normal" or contains "Body", (iii) **Other/Empty**: everything else including blank paragraphs. Print the count of each group.

**(c)** For every Body paragraph with more than 5 words, compute its word count and character count excluding spaces. Print: (i) the paragraph with the highest word count, (ii) the paragraph with the lowest word count above 5 words, and (iii) the mean word count across all qualifying body paragraphs rounded to 2 decimal places.

**(d)** Build a nested dictionary structured as `{ style_name: { "count": N, "texts": [ list ] } }` for all styles found. Sort the outer dictionary by count in descending order. If a style has more than 2 paragraphs, show only the first 2 texts. Pretty-print the result.

In [ ]:
# (a)
doc4 = docx.Document(DOCX_A)
para_list = [{"text": p.text, "style": p.style.name} for p in doc4.paragraphs]
print("Total paragraphs:", len(para_list))
print("Unique styles:", sorted({d["style"] for d in para_list}))

In [ ]:
# (b)
headings, body, other = [], [], []
for d in para_list:
    if d["text"].strip() == "":
        other.append(d)                                   # blank paragraphs go to Other/Empty
    elif d["style"].startswith("Heading"):
        headings.append(d)
    elif d["style"] == "Normal" or "Body" in d["style"]:
        body.append(d)
    else:
        other.append(d)
print("Headings    :", len(headings))
print("Body        :", len(body))
print("Other/Empty :", len(other))

In [ ]:
# (c)
qualifying = []
for d in body:
    wc = len(d["text"].split())
    if wc > 5:
        qualifying.append({"text": d["text"], "words": wc, "chars": len(d["text"].replace(" ", ""))})

if qualifying:
    hi = max(qualifying, key=lambda x: x["words"])
    lo = min(qualifying, key=lambda x: x["words"])
    mean_wc = round(sum(x["words"] for x in qualifying) / len(qualifying), 2)
    print(f"(i) Highest ({hi['words']} words, {hi['chars']} chars): {hi['text']}")
    print(f"(ii) Lowest  ({lo['words']} words, {lo['chars']} chars): {lo['text']}")
    print(f"(iii) Mean word count: {mean_wc}")
else:
    print("No body paragraphs with more than 5 words.")

In [ ]:
# (d)
style_map = {}
for d in para_list:
    entry = style_map.setdefault(d["style"], {"count": 0, "texts": []})
    entry["count"] += 1
    entry["texts"].append(d["text"])

sorted_styles = dict(sorted(style_map.items(), key=lambda kv: kv[1]["count"], reverse=True))
for info in sorted_styles.values():
    if info["count"] > 2:
        info["texts"] = info["texts"][:2]       # show only the first 2 texts
pprint.pprint(sorted_styles, width=100, sort_dicts=False)

## Task 5: DOCX Table Extraction and Pandas Integration

**(a)** Open a `.docx` file that contains at least one table. Iterate through `doc.tables` and extract each table as a list of lists: `[[row1col1, row1col2, ...], [row2col1, ...]]`. Print the total number of tables found and the dimensions (rows × columns) of each.

**(b)** Take the first table's data, treat the first row as the header and the remaining rows as data, and create a pandas DataFrame. Print the full DataFrame. Then count and print the number of empty or `None` cells per column.

**(c)** Extract all paragraphs that appear before the first table in the document by iterating through `doc.element.body` children and stopping once a `w:tbl` element is encountered. Store these paragraphs in a list and print their count.

**(d)** Write both the pre-table paragraphs and the first table's data into a single file named `docx_extracted.txt`. The file must have clearly labeled sections: a `--- PRE-TABLE PARAGRAPHS ---` header followed by the paragraphs, then a `--- TABLE DATA ---` header followed by rows formatted as pipe-separated values, e.g., `col1 | col2 | col3`.

In [ ]:
# (a)
doc5 = docx.Document(DOCX_A)
tables_data = []
for tbl in doc5.tables:
    tables_data.append([[cell.text for cell in row.cells] for row in tbl.rows])
print("Total tables found:", len(tables_data))
for i, t_ in enumerate(tables_data, start=1):
    print(f"Table {i}: {len(t_)} rows x {len(t_[0]) if t_ else 0} columns")

In [ ]:
# (b)
first = tables_data[0]
df5 = pd.DataFrame(first[1:], columns=first[0])
print(df5)
empty_counts = df5.apply(lambda col: col.map(lambda v: v is None or str(v).strip() == "").sum())
print("\nEmpty/None cells per column:")
print(empty_counts)

In [ ]:
# (c)
pre_table_paras = []
for child in doc5.element.body.iterchildren():
    if child.tag.endswith("}tbl"):
        break
    if child.tag.endswith("}p"):
        pre_table_paras.append(Paragraph(child, doc5).text)
print("Paragraphs before the first table:", len(pre_table_paras))

In [ ]:
# (d)
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for p in pre_table_paras:
        f.write(p + "\n")
    f.write("\n--- TABLE DATA ---\n")
    for row in first:
        f.write(" | ".join(row) + "\n")
print(open("docx_extracted.txt", encoding="utf-8").read())

## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

**(a)** Open two different `.docx` files (Document A and Document B). From each, extract all paragraphs whose style name starts with "Heading" into two separate lists. Using Python set operations, print: (i) headings common to both documents, (ii) headings only in Document A, and (iii) headings only in Document B.

**(b)** From each document, extract all "Normal" style paragraphs. Merge both lists. Remove exact duplicate paragraphs by comparing stripped, lowercased text. Print the total count before and after deduplication.

**(c)** Tokenize every paragraph in the deduplicated list into words (lowercase, punctuation removed via `re.sub()`). Build a combined word frequency dictionary using `collections.Counter`. Remove at least 15 stopwords. Display the top 20 most frequent words.

**(d)** Compute and print a summary containing: (i) total headings across both documents, (ii) total unique body paragraphs after deduplication, (iii) total unique word types (vocabulary size), (iv) average paragraph length in words rounded to 2 decimal places, and (v) the single longest paragraph by word count from the combined set.

In [ ]:
# (a)
docA, docB = docx.Document(DOCX_A), docx.Document(DOCX_B)
heads_A = [p.text for p in docA.paragraphs if p.style.name.startswith("Heading")]
heads_B = [p.text for p in docB.paragraphs if p.style.name.startswith("Heading")]
print("(i) Common headings:   ", sorted(set(heads_A) & set(heads_B)))
print("(ii) Only in Document A:", sorted(set(heads_A) - set(heads_B)))
print("(iii) Only in Document B:", sorted(set(heads_B) - set(heads_A)))

In [ ]:
# (b)
normal_A = [p.text for p in docA.paragraphs if p.style.name == "Normal" and p.text.strip()]
normal_B = [p.text for p in docB.paragraphs if p.style.name == "Normal" and p.text.strip()]
merged = normal_A + normal_B
print("Total before deduplication:", len(merged))

seen, dedup = set(), []
for para in merged:
    key = para.strip().lower()
    if key not in seen:
        seen.add(key)
        dedup.append(para)
print("Total after deduplication: ", len(dedup))

In [ ]:
# (c)
stop_t6 = {"the","a","an","and","of","to","in","is","are","was","were","that","this","it","for","on",
           "with","as","by","at","be","or","we","each","into","can","should","more","all"}
counter = Counter()
for para in dedup:
    words = re.sub(r"[^\w\s]|\d|_", "", para.lower()).split()
    counter.update(w for w in words if w not in stop_t6)
print("Top 20 words:")
for w, c in counter.most_common(20):
    print(f"  {w:<15}{c}")

In [ ]:
# (d)
word_counts = [len(p.split()) for p in dedup]
longest = max(dedup, key=lambda p: len(p.split()))
print("(i)   Total headings across both docs:", len(heads_A) + len(heads_B))
print("(ii)  Unique body paragraphs:         ", len(dedup))
print("(iii) Vocabulary size (word types):   ", len(counter))
print("(iv)  Average paragraph length (words):", round(sum(word_counts) / len(word_counts), 2))
print(f"(v)   Longest paragraph ({len(longest.split())} words):\n      {longest}")

## Task 7: Multi-Endpoint API Data Joining and Analysis

**(a)** Send GET requests to both endpoints below. For each, verify the HTTP status code is exactly 200; if not, raise a `ConnectionError` with the actual status code in the message. Parse both responses as JSON lists.
- https://jsonplaceholder.typicode.com/users
- https://jsonplaceholder.typicode.com/posts

**(b)** Join the two datasets using `id` from users and `userId` from posts. Build a dictionary structured as: `{ user_name: { "email": "...", "post_count": N, "titles": [ list of post titles ] } }`. Print the result for the first 3 users only.

**(c)** From the joined dictionary, find the user who has written the most posts. Print their: (i) full name, (ii) email address, (iii) company name, and (iv) all post titles numbered.

**(d)** Filter users where either of the following is true: (i) `company.bs` field contains the word "synergies" (case-insensitive), or (ii) `address.city` has more than 8 characters. Print the name, city, and post count for all matching users.

In [ ]:
# (a)
BASE = "https://jsonplaceholder.typicode.com"

def get_json_strict(url):
    r = requests.get(url, timeout=15)
    if r.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {r.status_code}")
    return r.json()

users7 = get_json_strict(f"{BASE}/users")
posts7 = get_json_strict(f"{BASE}/posts")
print(f"Fetched {len(users7)} users and {len(posts7)} posts")

In [ ]:
# (b)
joined = {}
for u in users7:
    user_posts = [p["title"] for p in posts7 if p["userId"] == u["id"]]
    joined[u["name"]] = {"email": u["email"], "post_count": len(user_posts), "titles": user_posts}

for name in list(joined)[:3]:
    print(name)
    pprint.pprint(joined[name], width=100, sort_dicts=False)
    print()

In [ ]:
# (c)
top_name = max(joined, key=lambda n: joined[n]["post_count"])
top_user = next(u for u in users7 if u["name"] == top_name)
print("(i)   Name:   ", top_name)
print("(ii)  Email:  ", joined[top_name]["email"])
print("(iii) Company:", top_user["company"]["name"])
print("(iv)  Post titles:")
for i, title in enumerate(joined[top_name]["titles"], start=1):
    print(f"   {i}. {title}")

In [ ]:
# (d)
print(f"{'Name':<28}{'City':<18}{'Posts'}")
for u in users7:
    cond1 = "synergies" in u["company"]["bs"].lower()
    cond2 = len(u["address"]["city"]) > 8
    if cond1 or cond2:
        print(f"{u['name']:<28}{u['address']['city']:<18}{joined[u['name']]['post_count']}")

## Task 8: To-Do Completion Rate Analysis Across Users

**(a)** Fetch data from https://jsonplaceholder.typicode.com/todos and parse it into a Python list. Validate that: (i) the response status is 200, (ii) the parsed result is a list, and (iii) the list contains at least one record. If any validation fails, print a meaningful error and exit. Print the total number of todo records retrieved.

**(b)** Group todos by `userId`. For each user, compute and store: `{ userId: { "total": N, "completed": N, "incomplete": N } }`. Print the result for all 10 users. For each user, verify that `completed + incomplete == total` and print `True` or `False` accordingly.

**(c)** Compute the completion rate for each user as `completed / total × 100` rounded to 1 decimal place. Print: (i) the `userId` with the highest completion rate and its value, (ii) the `userId` with the lowest completion rate and its value, and (iii) the overall average completion rate across all users.

**(d)** Fetch https://jsonplaceholder.typicode.com/users and build a lookup dictionary `{ userId: user_name }`. Print a formatted table with columns: `Rank | Name | Total Tasks | Completed | Completion Rate %`, sorted by completion rate in descending order. Use f-strings or `str.ljust()` to align columns neatly.

In [ ]:
# (a)
try:
    resp = requests.get(f"{BASE}/todos", timeout=15)
except requests.RequestException as e:
    print("Error: could not reach the server:", e); sys.exit(1)
if resp.status_code != 200:
    print(f"Error: expected status 200 but got {resp.status_code}"); sys.exit(1)
todos = resp.json()
if not isinstance(todos, list):
    print("Error: response is not a list"); sys.exit(1)
if len(todos) < 1:
    print("Error: the list contains no records"); sys.exit(1)
print("Total todo records retrieved:", len(todos))

In [ ]:
# (b)
stats = {}
for t_ in todos:
    s = stats.setdefault(t_["userId"], {"total": 0, "completed": 0, "incomplete": 0})
    s["total"] += 1
    s["completed" if t_["completed"] else "incomplete"] += 1

for uid in sorted(stats):
    s = stats[uid]
    print(f"User {uid}: {s}  ->  completed + incomplete == total: {s['completed'] + s['incomplete'] == s['total']}")

In [ ]:
# (c)
rates = {uid: round(s["completed"] / s["total"] * 100, 1) for uid, s in stats.items()}
best = max(rates, key=rates.get)
worst = min(rates, key=rates.get)
print(f"(i)   Highest completion rate: userId {best} -> {rates[best]}%")
print(f"(ii)  Lowest completion rate:  userId {worst} -> {rates[worst]}%")
print(f"(iii) Overall average rate:    {round(sum(rates.values()) / len(rates), 1)}%")

In [ ]:
# (d)
users8 = get_json_strict(f"{BASE}/users")
lookup = {u["id"]: u["name"] for u in users8}

ranked = sorted(rates.items(), key=lambda kv: kv[1], reverse=True)
print(f"{'Rank':<6}{'Name':<26}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %'}")
print("-" * 76)
for rank, (uid, rate) in enumerate(ranked, start=1):
    print(f"{rank:<6}{lookup.get(uid, 'Unknown'):<26}{stats[uid]['total']:<14}{stats[uid]['completed']:<12}{rate}")

## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

**(a)** Fetch data from https://jsonplaceholder.typicode.com/users. Each user contains nested objects (`address` with `geo`, and `company`). Write a function `flatten_user(user: dict) -> dict` that returns a single-level dictionary with exactly these keys: `id, name, username, email, city, zipcode, lat, lng, company_name, company_catchphrase`. Apply this function to all 10 users and print the resulting flattened list.

**(b)** Create a pandas DataFrame from the flattened user list and display it in full. Then: (i) cast the `lat` and `lng` columns to float, (ii) add a new column `distance_from_origin` computed as `sqrt(lat² + lng²)` using the `math` module applied row-by-row, and (iii) print the user with the greatest distance from origin.

**(c)** Using the `lat` column, classify users into two groups: (i) Northern Hemisphere where `lat > 0`, and (ii) Southern Hemisphere where `lat < 0`. Print the names of users in each group. Then determine which hemisphere has the higher average `distance_from_origin`.

**(d)** Fetch both endpoints below. For each post, count its number of comments by matching `postId` in comments against `id` in posts. Build a sorted list of tuples: `(post_id, post_title[:40], comment_count)` in descending order by comment count. Print the top 5 most-commented and bottom 5 least-commented posts.
- https://jsonplaceholder.typicode.com/posts
- https://jsonplaceholder.typicode.com/comments

In [ ]:
# (a)
def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }

users9 = get_json_strict(f"{BASE}/users")
flat_users = [flatten_user(u) for u in users9]
pprint.pprint(flat_users, width=110, sort_dicts=False)

In [ ]:
# (b)
df9 = pd.DataFrame(flat_users)
with pd.option_context("display.max_columns", None, "display.width", 250):
    display(df9)

df9["lat"] = df9["lat"].astype(float)
df9["lng"] = df9["lng"].astype(float)
df9["distance_from_origin"] = df9.apply(lambda r: math.sqrt(r["lat"] ** 2 + r["lng"] ** 2), axis=1)

far = df9.loc[df9["distance_from_origin"].idxmax()]
print(f"User farthest from origin: {far['name']} (distance = {far['distance_from_origin']:.2f})")

In [ ]:
# (c)
north = df9[df9["lat"] > 0]
south = df9[df9["lat"] < 0]
print("Northern Hemisphere:", north["name"].tolist())
print("Southern Hemisphere:", south["name"].tolist())

avg_n = north["distance_from_origin"].mean() if len(north) else float("nan")
avg_s = south["distance_from_origin"].mean() if len(south) else float("nan")
print(f"Average distance - North: {avg_n:.2f} | South: {avg_s:.2f}")
print("Higher average distance:", "Northern Hemisphere" if avg_n > avg_s else "Southern Hemisphere")

In [ ]:
# (d)
posts9 = get_json_strict(f"{BASE}/posts")
comments9 = get_json_strict(f"{BASE}/comments")

comment_counts = Counter(c["postId"] for c in comments9)
post_stats = [(p["id"], p["title"][:40], comment_counts.get(p["id"], 0)) for p in posts9]
post_stats.sort(key=lambda t: t[2], reverse=True)

print("Top 5 most-commented posts:")
for t_ in post_stats[:5]:
    print("  ", t_)
print("\nBottom 5 least-commented posts:")
for t_ in post_stats[-5:]:
    print("  ", t_)